# PyCAM-SIMA Python-owned FKESSLER Notebook

This is the single maintained PyCAM-SIMA Notebook. A normal one-process Jupyter kernel controls 24 Python MPI workers through the existing authenticated socket/PBS controller.

There is one model path: `start()` performs pure-Python initialization, every persistent state array is owned by NumPy, and the Fortran shared library contains only stateless numerical kernels. The old cam_init/cam_run wrapper backend has been removed.

## Runtime architecture

```text
Jupyter controller
        │ authenticated socket / PBS
        ▼
24 Python MPI workers (one process per rank)
        ├── read YAML and atm_in
        ├── build cubed-sphere grid and rank-local partition
        ├── allocate every persistent NumPy array
        ├── read vertical coordinates and generate DCMIP2016 state
        ├── initialize clock, constituents, and parameters
        ├── communicate through mpi4py
        └── Python StatePool: T/U/V/Q/PS, tendencies, grid, process state
                          │ explicit zero-copy array arguments
                          ▼
              stateless Fortran numerical kernel .so
```

Rank 0 reads fixed NetCDF inputs and broadcasts them with `mpi4py`; no CAM initialization or driver routine is called. The shared library is loaded before initialization, but its ABI and numerical functions are first touched only at an explicit computational phase.

## 1. Fixed v1 configuration

The model is intentionally fixed to CAM-SIMA `f8daa568eae2696b7c4ebff7768f02f5d097d9df`, FKESSLER, ne3np4.pg3, L30, 24 MPI ranks, one thread per rank, a 1800 s timestep, and the DCMIP2016 moist baroclinic wave.

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import os
import shutil

import pycam_sima
from pycam_sima import ModelConfig, NotebookSession
from pycam_sima.model.validation import (
    compare_history_directories,
    compare_history_files,
)

repo = Path('/glade/work/ruitong/pycam-sima')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
config_path = repo / 'configs/fkessler_model.yaml'
config = ModelConfig.from_yaml(config_path)
reference_atm_in = (
    repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
)
oracle_path = os.environ.get('PYCAM_SIMA_ORACLE_DIR')
oracle_run = Path(oracle_path).expanduser().resolve() if oracle_path else None

stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
run_dir = scratch / 'pycam-sima/notebook_trials' / f'model-{stamp}' / 'run'
run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, run_dir / 'atm_in')
history_dir = run_dir / 'history'

print('pycam_sima', pycam_sima.__version__)
print('run directory:', run_dir)
config.as_dict()

## 2. Start 24 Python MPI workers

On a Derecho login-node kernel, `start()` submits a PBS worker; inside an allocation it launches locally. It returns in state `INITIALIZED` before any model/kernel function or ABI version function has been called.

In [ ]:
if 'model' in globals() and model.running:
    model.close()

model = NotebookSession(
    config_path,
    run_dir=run_dir,
    history_dir=history_dir,
    python_executable=repo / '.venv/bin/python',
    log_path=run_dir / 'mpi-worker.log',
)
model.start()
assert model.initialized_native_calls == 0
assert model.initialized_abi_checked is False
print({
    'mode': model.launch_mode_used,
    'job': model.job_id,
    'ranks': model.ranks,
    'fields_and_aliases': len(model.field_names),
    'phase_status': model.phase_status,
    'scheme_status': model.scheme_status,
})

## 3. Inspect Python-owned fields

`parameters.field(name)` returns a typed remote handle. `get()` transfers a rank-local copy to the Notebook; `stats()` computes a compact summary in the MPI worker. Canonical state metadata records owner, intent, dimensions, units, lifetime, category, and alias status.

In [ ]:
temperature_field = model.parameters.air_temperature
print(temperature_field.info)
print(temperature_field.stats(rank=0))
temperature_rank0 = temperature_field.get(rank=0)
temperature_rank0

In [ ]:
parameter_summary = model.parameters.describe()
assert parameter_summary['runtime']['state_owner'] == 'python'
assert all(model.field_info(name)['owner'] == 'python' for name in model.field_names)
print('all fields and zero-copy aliases:', len(parameter_summary['all_fields']))
parameter_summary['key_fields']

## 4. Explicit nstep=0 preparation

`prepare_initial_step()` executes dynamics-to-physics mapping, physics timestep initialization, the FKESSLER before-coupler schemes, and writes nstep=0 history. Calling `model.step()` directly from `INITIALIZED` performs this preparation automatically.

In [ ]:
status = model.prepare_initial_step()
print(status)
print('history files:', len(list(history_dir.glob('*.nc'))))

## 5. Inspect and run individual CCPP schemes

The Kessler CCPP suite is no longer a single opaque before/after block. All 19 `physics_before_coupler` schemes and all 5 `physics_after_coupler` schemes are separate Python interfaces. Every call is collective across all 24 workers, and every return checks that Python array addresses are unchanged.

In [ ]:
{
    'model_phases': model.phase_names,
    'before_coupler': model.scheme_plan.describe('physics_before_coupler'),
    'after_coupler': model.scheme_plan.describe('physics_after_coupler'),
}

In [ ]:
# Example for a fresh scheme-by-scheme session:
# model.run_phase('dynamics_to_physics')
# model.run_phase('physics_timestep_initial')
# model.run_scheme('calc_exner', group='physics_before_coupler')
# exner_after_scheme = model.parameters.field('exner_function').get(rank=0)
# model.run_scheme('kessler', group='physics_before_coupler')
# state_after_kessler = model.parameters.physics_air_temperature.get(rank=0)

# Intentional non-validated control experiments require unsafe=True:
# model.scheme_plan.disable('kessler_diagnostics', unsafe=True)
# model.scheme_plan.move('kessler', after='kessler_update', unsafe=True)
# model.scheme_plan.reset()  # restore the BFB-validated XML order

model.scheme_plan.sequence_safe

## 6. Optional field modification

Prognostic, tendency, and process fields may be changed at a Python boundary without replacing their NumPy storage. Static grid/topology fields reject writes unless `unsafe=True` is explicit. Any numerical edit intentionally breaks BFB.

In [ ]:
# changed = temperature_field.get(rank=0)
# changed[0, 0, 0, 0, 0] += 1.0e-6
# temperature_field.set(changed, rank=0)
# print(temperature_field.stats(rank=0))

# Static experiment, deliberately unsafe:
# gll = model.parameters.field('gll_node').get(rank=0)
# model.parameters.field('gll_node').set(gll, rank=0, unsafe=True)

## 7. Advance one complete 1800 s timestep

In [ ]:
step = model.step()
print('completed step:', step)
print(temperature_field.stats(rank=0))
print('history files:', len(list(history_dir.glob('*.nc'))))

## 8. Optional complete 50-step run

Run this only with untouched fields. `step(count)` remains one socket request while all 24 workers execute the same fixed plan.

In [ ]:
# if model.current_step < config.stop_n:
#     model.step(config.stop_n - model.current_step)
# print(model.current_step, len(list(history_dir.glob('*.nc'))))

## 9. Finalize and optionally compare history

The model run is self-contained. To compare against output from the pinned external CAM-SIMA executable, set `PYCAM_SIMA_ORACLE_DIR` to its history directory before running the setup cell.

In [ ]:
model.close()
print('closed:', run_dir)

In [ ]:
candidate_files = sorted(history_dir.glob('*.nc'))
if oracle_run is None:
    print('External comparison skipped; set PYCAM_SIMA_ORACLE_DIR to enable it')
else:
    for candidate in candidate_files:
        compare_history_files(oracle_run / candidate.name, candidate)
    print(f'BFB for all {len(candidate_files)} available model timestamps')

    if len(candidate_files) == 51:
        compare_history_directories(
            oracle_run, history_dir,
            expected_files=51, expected_numeric_variables=26,
        )
        print('FULL BFB: 50 steps, 51 timestamps, 26 numeric variables')

## 10. Recorded full validation

After the repository-level 50-step gate has run, its immutable evidence is stored in `validation/fkessler_model_bfb.json`.

In [ ]:
evidence_path = repo / 'validation/fkessler_model_bfb.json'
json.loads(evidence_path.read_text()) if evidence_path.exists() else 'validation pending'